# 复现实验：轻量级 LLM 事故响应（NDSS 2026）

本 notebook 在 **Google Colab 免费 T4 GPU** 上运行论文《Incident Response Planning Using a Lightweight Large Language Model with Reduced Hallucination》（Hammar, Alpcan, Lupu, NDSS 2026）附录 F 的四个复现步骤（E1–E4），并验证论文五项声明（C1–C5）。

**使用方法**：菜单栏 `代码执行程序` → `更改运行时类型` → 硬件加速器选 **T4 GPU** → 保存，然后 `代码执行程序` → `全部运行`（首次约 30–50 分钟，大头是下载约 15GB 模型权重）。

| Notebook 步骤 | 论文实验 | 验证的声明 | 论文位置 |
|---|---|---|---|
| 第 2 格 | (S1) 加载微调 LLM | C2、C3 | 附录 F，第 19–20 页 |
| 第 3 格 | (S2) 加载训练数据集 | C1 | 附录 F，第 20 页 |
| 第 4 格 | (S3) 生成响应动作 | C3、C5 | 附录 F，第 20 页 |
| 第 5 格 | (S4) 微调（缩小版） | 训练流程演示 | 附录 F，第 20 页 |

**与论文原设置的两处差异（为适配 T4 显存/精度，功能不受影响）**：
1. 论文用 fp16 半精度加载（需约 28GB 显存，T4 装不下）；本 notebook 按**论文表 7 已采用的 4-bit NF4 量化**加载（约 9GB，T4 可容纳），计算精度用 fp16（T4 不支持 bf16，论文训练用的是 bf16）。
2. (S4) 与论文一样只取 5 条样本做功能演示（论文实验 E4 的设计），并额外挑选最短的 5 条以保证 16GB 显存安全。

**对应仓库**：https://github.com/zjf134/llm_incident_response_ndss26 （含中文注释版代码与本指南）

In [ ]:
# ── 第 1 格：环境检查与安装（对应论文附录 F-B「安装与配置」）──
# 预期：显示 Tesla T4 / 16GB 显存。若报错无 GPU，请先：代码执行程序 → 更改运行时类型 → T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 安装论文工件依赖（锁版本，首次约 3-8 分钟；若提示"重启会话"，重启后重跑本格即可，权重缓存不丢）
!pip install -q llm_recovery==0.0.13

import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU 显存: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
# ── 第 2 格 (S1)：以 4-bit 量化加载微调好的 LLM（论文实验 E1/E2，验证 C2/C3）──
# 期望输出：LLM loaded ... | 显存占用约 9 GB（对照论文 E1 期望输出 "LLM loaded successfully."）
# 说明：模型 = DeepSeek-R1-Distill-Qwen-14B 基座 + 事故响应 LoRA 适配器（约 201MB，自动加载），
#       即论文 Remark 1 的 14B 轻量模型；4-bit NF4 + 双重量化与论文表 7 的量化设定一致。
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "kimhammar/LLMIncidentResponse"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,       # 双重量化（论文训练设定）
    bnb_4bit_quant_type="nf4",            # NF4（论文训练设定）
    bnb_4bit_compute_dtype=torch.float16, # T4 不支持 bf16，故用 fp16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print("LLM loaded on:", model.device, "| 已用显存: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# ── 第 3 格 (S2)：加载 68,000 条事故训练数据集（论文实验 E2，验证 C1）──
# 期望输出：样本总数 + 第一条指令样例（结构即论文附录 B 的提示词模板）
# 指令模板结构：### System 系统描述 / ### Logs 日志 / ### Incident 事故摘要 /
#               ### State 六维恢复状态 / ### Previous recovery actions 已执行动作 / ### Instruction 任务要求
from datasets import load_dataset

ds = load_dataset("kimhammar/CSLE-IncidentResponse-V1", data_files="examples_16_june.json")
instructions = list(ds["train"]["instructions"][0])
answers = list(ds["train"]["answers"][0])
print("Training dataset loaded successfully. Number of instructions: %d. Number of answers: %d."
      % (len(instructions), len(answers)))
print()
print(instructions[0][:2000])

In [ ]:
# ── 第 4 格 (S3)：用微调模型生成事故响应动作（论文实验 E3，验证 C3/C5）──
# 期望输出：先 <think> 链式思考，再输出 {"Action": ..., "Explanation": ...}（对照论文第 20 页 E3 示例）
# 速度：T4 上约 5-10 token/s，通常几百 token 内停止（数分钟）；6000 为上限，可随时中断
import random
from threading import Thread
from transformers import TextIteratorStreamer

instruction = random.choice(instructions)
inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
gen_kwargs = dict(max_new_tokens=6000, temperature=0.8, do_sample=True)  # 与论文脚本一致

streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)
thread = Thread(target=model.generate, kwargs={**inputs, **gen_kwargs, "streamer": streamer})
thread.start()
for new_text in streamer:
    print(new_text, end="", flush=True)
thread.join()

In [ ]:
# ── 第 5 格 (S4)：QLoRA 微调缩小版（论文实验 E4，演示训练流程）──
# 期望输出：Trainable parameters: 50331648（与论文一致）+ "Step: 1, Epoch: 0.5000, ..." 日志
# 与论文脚本的差异仅两处：① 挑最短的 5 条样本（T4 显存安全）；② bf16=True 改为 fp16=True（T4 不支持 bf16）
# 其余超参数与论文表 7 完全一致：LoRA r=64, alpha=128, dropout=0.05, lr=0.00095, batch=1
from transformers import Trainer, TrainingArguments, PrinterCallback, ProgressCallback
from llm_recovery.fine_tuning.lora import LORA
from llm_recovery.fine_tuning.examples_dataset import ExamplesDataset
from llm_recovery.fine_tuning.logging_callback import LoggingCallback

# 1) 挑选 5 条最短的指令-答案对（预分词按长度排序，避免 T4 显存溢出）
lens = [len(tokenizer(x)["input_ids"]) + len(tokenizer(y)["input_ids"])
        for x, y in zip(instructions, answers)]
order = sorted(range(len(instructions)), key=lambda i: lens[i])[:5]
short_instructions = [instructions[i] for i in order]
short_answers = [answers[i] for i in order]

# 2) 用第 2 格已加载的 4-bit 模型准备 QLoRA（内部自动 prepare_model_for_kbit_training + 梯度检查点）
seed = 99125
llm = LORA.setup_llm_for_fine_tuning(llm=model, r=64, lora_alpha=128, lora_dropout=0.05)
ft_dataset = ExamplesDataset(instructions=short_instructions, answers=short_answers, tokenizer=tokenizer)

# 3) 训练循环：复刻 llm_recovery.LORA.supervised_fine_tuning，仅把 bf16=True 换成 fp16=True
args = TrainingArguments(
    output_dir="ds-dt-lora", fp16=True,
    per_device_train_batch_size=1, num_train_epochs=1, learning_rate=0.00095,
    logging_steps=1, save_strategy="steps", save_steps=25, save_total_limit=3,
    gradient_accumulation_steps=1, seed=seed, report_to=[])
callback = LoggingCallback(
    prompts=short_instructions, answers=short_answers, tokenizer=tokenizer, dataset=ft_dataset,
    window=100, gen_kwargs=dict(max_new_tokens=6000, temperature=0.6, do_sample=True),
    prompt_logging=False, prompt_logging_frequency=50, progress_save_frequency=10, seed=seed)
trainer = Trainer(model=llm, args=args, train_dataset=ft_dataset,
                  data_collator=ft_dataset.collate, callbacks=[callback])
trainer.remove_callback(PrinterCallback)
trainer.remove_callback(ProgressCallback)
trainer.train()
trainer.save_model(output_dir="ds-dt-lora")
print("微调完成，LoRA 适配器已保存到 ./ds-dt-lora")

## 结果对照与排障

**期望输出对照（论文第 20 页）**：
- (S1)/(S2)：`LLM loaded successfully` / `Training dataset loaded successfully. Number of instructions: ...`
- (S3)：`<think>` 推理 + JSON 动作，例如 `"Action": "Acquire full disk and memory images of 10.20.11.42 ..."`（注意温度 0.8 采样，每次输出不同）
- (S4)：`Trainable parameters: 50331648`、`Step: 1, Epoch: 0.5000, Avg_loss=..., LR=...`

**常见问题**：
- **显存溢出 (OOM)**：运行时菜单 `代码执行程序` → `断开连接并删除运行时`，重新连接 T4 后按格重跑；(S4) 已默认挑最短样本，仍溢出可减为 3 条。
- **想跑全量微调**：把第 5 格的 `order[:5]` 改为全量 `order = list(range(len(instructions)))`，但免费 T4 需数天，建议改用 Kaggle（2×T4，每周 30 小时）或租卡（AutoDL 4090 约 2 元/小时）。
- **下载慢/失败**：Colab 在境外直连 Hugging Face，一般无需镜像；若遇网络抖动重跑该格即可（断点续传）。
- **与论文数字的关系**：本 notebook 复现的是论文附录 F 的工件验证实验（E1–E4）；§VII 的完整基准对比（25 起事故 × 5 种子、与 O3/Gemini/PPO 对比）需要前沿模型 API 与 CSLE 框架，请直接引用论文图 12–16 的数字。

**引用**：Hammar K, Alpcan T, Lupu E C. Incident response planning using a lightweight large language model with reduced hallucination. NDSS 2026. DOI: 10.14722/ndss.2026.240358（原仓库：Kim-Hammar/llm_incident_response_ndss26，CC-BY-SA 4.0）